# 01 - Explore CBIS-DDSM metadata (A2)

Goal: build one clean table (one row per ROI crop) for `~/data/cbis-ddsm/processed/rois.csv`.

Data lives outside the repo at `~/data/cbis-ddsm/` (`csv/` + `jpeg/`). Kaggle: "CBIS-DDSM: Breast Cancer Image Dataset" (awsaf49), JPEG version.

In [7]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

DATA = Path.home() / "data" / "cbis-ddsm"
CSV = DATA / "csv"
JPEG = DATA / "jpeg"

# sanity check: do the paths exist?
print(CSV.exists(), JPEG.exists())
print(sorted(p.name for p in CSV.iterdir()))

True True
['calc_case_description_test_set.csv', 'calc_case_description_train_set.csv', 'dicom_info.csv', 'mass_case_description_test_set.csv', 'mass_case_description_train_set.csv', 'meta.csv']


## 1. Case description CSVs (mass / calc x train / test)

Findings:
- Mass files have `mass shape`, `mass margins`; calc files have `calc type`, `calc distribution`. The other 12 columns are the same.
- Density column name differs: `breast_density` (mass) vs `breast density` (calc). Must standardize before concatenating.
- Path columns: `image file path`, `cropped image file path`, `ROI mask file path` (the last three columns).

In [9]:
# load the four case files into separate DataFrames
mass_train = pd.read_csv(CSV / 'mass_case_description_train_set.csv')
mass_test = pd.read_csv(CSV / 'mass_case_description_test_set.csv')
calc_train = pd.read_csv(CSV / 'calc_case_description_train_set.csv')
calc_test = pd.read_csv(CSV / 'calc_case_description_test_set.csv')

# for each of the four, print shape, columns, and 3 rows
for name, case_df in [('mass_train', mass_train), ('mass_test', mass_test),
                      ('calc_train', calc_train), ('calc_test', calc_test)]:
    print(name, case_df.shape)
    print(case_df.columns.tolist())
    display(case_df.head(3))

mass_train (1318, 14)
['patient_id', 'breast_density', 'left or right breast', 'image view', 'abnormality id', 'abnormality type', 'mass shape', 'mass margins', 'assessment', 'pathology', 'subtlety', 'image file path', 'cropped image file path', 'ROI mask file path']


,patient_id,breast_density,left or right breast,image view,abnormality id,abnormality type,mass shape,mass margins,assessment,pathology,subtlety,image file path,cropped image file path,ROI mask file path
0,P_00001,3,LEFT,CC,1,mass,IRREGULAR-ARCHITECTURAL_DISTORTION,SPICULATED,4,MALIGNANT,4,Mass-Training_P_00001_LEFT_CC/1.3.6.1.4.1.9590...,Mass-Training_P_00001_LEFT_CC_1/1.3.6.1.4.1.95...,Mass-Training_P_00001_LEFT_CC_1/1.3.6.1.4.1.95...
1,P_00001,3,LEFT,MLO,1,mass,IRREGULAR-ARCHITECTURAL_DISTORTION,SPICULATED,4,MALIGNANT,4,Mass-Training_P_00001_LEFT_MLO/1.3.6.1.4.1.959...,Mass-Training_P_00001_LEFT_MLO_1/1.3.6.1.4.1.9...,Mass-Training_P_00001_LEFT_MLO_1/1.3.6.1.4.1.9...
2,P_00004,3,LEFT,CC,1,mass,ARCHITECTURAL_DISTORTION,ILL_DEFINED,4,BENIGN,3,Mass-Training_P_00004_LEFT_CC/1.3.6.1.4.1.9590...,Mass-Training_P_00004_LEFT_CC_1/1.3.6.1.4.1.95...,Mass-Training_P_00004_LEFT_CC_1/1.3.6.1.4.1.95...


mass_test (378, 14)
['patient_id', 'breast_density', 'left or right breast', 'image view', 'abnormality id', 'abnormality type', 'mass shape', 'mass margins', 'assessment', 'pathology', 'subtlety', 'image file path', 'cropped image file path', 'ROI mask file path']


,patient_id,breast_density,left or right breast,image view,abnormality id,abnormality type,mass shape,mass margins,assessment,pathology,subtlety,image file path,cropped image file path,ROI mask file path
0,P_00016,4,LEFT,CC,1,mass,IRREGULAR,SPICULATED,5,MALIGNANT,5,Mass-Test_P_00016_LEFT_CC/1.3.6.1.4.1.9590.100...,Mass-Test_P_00016_LEFT_CC_1/1.3.6.1.4.1.9590.1...,Mass-Test_P_00016_LEFT_CC_1/1.3.6.1.4.1.9590.1...
1,P_00016,4,LEFT,MLO,1,mass,IRREGULAR,SPICULATED,5,MALIGNANT,5,Mass-Test_P_00016_LEFT_MLO/1.3.6.1.4.1.9590.10...,Mass-Test_P_00016_LEFT_MLO_1/1.3.6.1.4.1.9590....,Mass-Test_P_00016_LEFT_MLO_1/1.3.6.1.4.1.9590....
2,P_00017,2,LEFT,CC,1,mass,ROUND,CIRCUMSCRIBED,4,MALIGNANT,4,Mass-Test_P_00017_LEFT_CC/1.3.6.1.4.1.9590.100...,Mass-Test_P_00017_LEFT_CC_1/1.3.6.1.4.1.9590.1...,Mass-Test_P_00017_LEFT_CC_1/1.3.6.1.4.1.9590.1...


calc_train (1546, 14)
['patient_id', 'breast density', 'left or right breast', 'image view', 'abnormality id', 'abnormality type', 'calc type', 'calc distribution', 'assessment', 'pathology', 'subtlety', 'image file path', 'cropped image file path', 'ROI mask file path']


,patient_id,breast density,left or right breast,image view,abnormality id,abnormality type,calc type,calc distribution,assessment,pathology,subtlety,image file path,cropped image file path,ROI mask file path
0,P_00005,3,RIGHT,CC,1,calcification,AMORPHOUS,CLUSTERED,3,MALIGNANT,3,Calc-Training_P_00005_RIGHT_CC/1.3.6.1.4.1.959...,Calc-Training_P_00005_RIGHT_CC_1/1.3.6.1.4.1.9...,Calc-Training_P_00005_RIGHT_CC_1/1.3.6.1.4.1.9...
1,P_00005,3,RIGHT,MLO,1,calcification,AMORPHOUS,CLUSTERED,3,MALIGNANT,3,Calc-Training_P_00005_RIGHT_MLO/1.3.6.1.4.1.95...,Calc-Training_P_00005_RIGHT_MLO_1/1.3.6.1.4.1....,Calc-Training_P_00005_RIGHT_MLO_1/1.3.6.1.4.1....
2,P_00007,4,LEFT,CC,1,calcification,PLEOMORPHIC,LINEAR,4,BENIGN,4,Calc-Training_P_00007_LEFT_CC/1.3.6.1.4.1.9590...,Calc-Training_P_00007_LEFT_CC_1/1.3.6.1.4.1.95...,Calc-Training_P_00007_LEFT_CC_1/1.3.6.1.4.1.95...


calc_test (326, 14)
['patient_id', 'breast density', 'left or right breast', 'image view', 'abnormality id', 'abnormality type', 'calc type', 'calc distribution', 'assessment', 'pathology', 'subtlety', 'image file path', 'cropped image file path', 'ROI mask file path']


,patient_id,breast density,left or right breast,image view,abnormality id,abnormality type,calc type,calc distribution,assessment,pathology,subtlety,image file path,cropped image file path,ROI mask file path
0,P_00038,2,LEFT,CC,1,calcification,PUNCTATE-PLEOMORPHIC,CLUSTERED,4,BENIGN,2,Calc-Test_P_00038_LEFT_CC/1.3.6.1.4.1.9590.100...,Calc-Test_P_00038_LEFT_CC_1/1.3.6.1.4.1.9590.1...,Calc-Test_P_00038_LEFT_CC_1/1.3.6.1.4.1.9590.1...
1,P_00038,2,LEFT,MLO,1,calcification,PUNCTATE-PLEOMORPHIC,CLUSTERED,4,BENIGN,2,Calc-Test_P_00038_LEFT_MLO/1.3.6.1.4.1.9590.10...,Calc-Test_P_00038_LEFT_MLO_1/1.3.6.1.4.1.9590....,Calc-Test_P_00038_LEFT_MLO_1/1.3.6.1.4.1.9590....
2,P_00038,2,RIGHT,CC,1,calcification,VASCULAR,NaN,2,BENIGN_WITHOUT_CALLBACK,5,Calc-Test_P_00038_RIGHT_CC/1.3.6.1.4.1.9590.10...,Calc-Test_P_00038_RIGHT_CC_1/1.3.6.1.4.1.9590....,Calc-Test_P_00038_RIGHT_CC_1/1.3.6.1.4.1.9590....


## 2. `dicom_info.csv`

Findings:
- 10,237 rows, one per JPEG. `image_path` looks like `CBIS-DDSM/jpeg/<SeriesUID>/<n>-<m>.jpg`; on this machine that is `~/data/cbis-ddsm/jpeg/<SeriesUID>/...`.
- `SeriesDescription` counts: cropped images 3567, ROI mask images 3247, full mammogram images 2857.
- 566 rows have a **blank** `SeriesDescription` (`value_counts()` hides them by default, which explains 9,671 vs 10,237).

In [10]:
dicom = pd.read_csv(CSV / 'dicom_info.csv')
print(dicom.shape, dicom.columns)
display(dicom.head(3))
print(dicom['SeriesDescription'].value_counts())
print(dicom['image_path'].head())

(10237, 38) Index(['file_path', 'image_path', 'AccessionNumber', 'BitsAllocated',
       'BitsStored', 'BodyPartExamined', 'Columns', 'ContentDate',
       'ContentTime', 'ConversionType', 'HighBit', 'InstanceNumber',
       'LargestImagePixelValue', 'Laterality', 'Modality', 'PatientBirthDate',
       'PatientID', 'PatientName', 'PatientOrientation', 'PatientSex',
       'PhotometricInterpretation', 'PixelRepresentation',
       'ReferringPhysicianName', 'Rows', 'SOPClassUID', 'SOPInstanceUID',
       'SamplesPerPixel', 'SecondaryCaptureDeviceManufacturer',
       'SecondaryCaptureDeviceManufacturerModelName', 'SeriesDescription',
       'SeriesInstanceUID', 'SeriesNumber', 'SmallestImagePixelValue',
       'SpecificCharacterSet', 'StudyDate', 'StudyID', 'StudyInstanceUID',
       'StudyTime'],
      dtype='str')


,file_path,image_path,AccessionNumber,BitsAllocated,BitsStored,BodyPartExamined,Columns,ContentDate,ContentTime,ConversionType,...,SecondaryCaptureDeviceManufacturerModelName,SeriesDescription,SeriesInstanceUID,SeriesNumber,SmallestImagePixelValue,SpecificCharacterSet,StudyDate,StudyID,StudyInstanceUID,StudyTime
0,CBIS-DDSM/dicom/1.3.6.1.4.1.9590.100.1.2.12930...,CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.129308...,NaN,16,16,BREAST,351,20160426,131732.685,WSD,...,MATLAB,cropped images,1.3.6.1.4.1.9590.100.1.2.129308726812851964007...,1,23078,ISO_IR 100,20160720.0,DDSM,1.3.6.1.4.1.9590.100.1.2.271867287611061855725...,214951.0
1,CBIS-DDSM/dicom/1.3.6.1.4.1.9590.100.1.2.24838...,CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.248386...,NaN,16,16,BREAST,3526,20160426,143829.101,WSD,...,MATLAB,full mammogram images,1.3.6.1.4.1.9590.100.1.2.248386742010678582309...,1,0,ISO_IR 100,20160720.0,DDSM,1.3.6.1.4.1.9590.100.1.2.161516517311681906612...,193426.0
2,CBIS-DDSM/dicom/1.3.6.1.4.1.9590.100.1.2.26721...,CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.267213...,NaN,16,16,BREAST,1546,20160503,111956.298,WSD,...,MATLAB,full mammogram images,1.3.6.1.4.1.9590.100.1.2.267213171011171858918...,1,0,ISO_IR 100,20160807.0,DDSM,1.3.6.1.4.1.9590.100.1.2.291043622711253836701...,161814.0


SeriesDescription
cropped images           3567
ROI mask images          3247
full mammogram images    2857
Name: count, dtype: int64
0    CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.129308...
1    CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.248386...
2    CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.267213...
3    CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.381187...
4    CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.381187...
Name: image_path, dtype: str


## 3. Mapping case-CSV paths to JPEGs

Case path format: `<patient folder>/<StudyUID>/<SeriesUID>/000000.dcm`. The **SeriesUID** (second to last part) matches `SeriesInstanceUID` in `dicom_info`.

Checked on `mass_train` row 0: the cropped path (`000000.dcm`) and the ROI mask path (`000001.dcm`) point to the **same series**, so joining on series UID returns two rows (one cropped, one ROI mask). They must be separated with `SeriesDescription` (JPEG file names like `1-250.jpg` / `2-249.jpg` do not match `000000`/`000001`).
For this case the crop is 515x431 and the mask is full-size 4808x3024; opening both images confirmed the labels.

In [11]:
print(mass_train.loc[0, 'cropped image file path'])
print(mass_train.loc[0, 'ROI mask file path'])

Mass-Training_P_00001_LEFT_CC_1/1.3.6.1.4.1.9590.100.1.2.108268213011361124203859148071588939106/1.3.6.1.4.1.9590.100.1.2.296736403313792599626368780122205399650/000000.dcm
Mass-Training_P_00001_LEFT_CC_1/1.3.6.1.4.1.9590.100.1.2.108268213011361124203859148071588939106/1.3.6.1.4.1.9590.100.1.2.296736403313792599626368780122205399650/000001.dcm



In [12]:
# 1. Pull the series UID out of the case path.
#    Path = <patient folder>/<StudyUID>/<SeriesUID>/000000.dcm, so the series UID is second to last.
crop_path = mass_train.loc[0, 'cropped image file path']
uid = crop_path.split('/')[-2]
print("series UID:", uid)

# 2. Find the matching rows in dicom_info.
match = dicom[dicom['SeriesInstanceUID'] == uid]
display(match[['image_path', 'SeriesDescription', 'Rows', 'Columns']])

# 3. Convert each CSV path to a real path on your Mac, then open and show the images.
fig, axes = plt.subplots(1, len(match), figsize=(10, 6))
for ax, (_, row) in zip(axes, match.iterrows()):
    jpg = JPEG / row['image_path'].replace('CBIS-DDSM/jpeg/', '')
    img = Image.open(jpg)
    print(row['SeriesDescription'], jpg.name, img.size, "exists:", jpg.exists())
    ax.imshow(img, cmap='gray')
    ax.set_title(f"{row['SeriesDescription']}\n{img.size}")
    ax.axis('off')
plt.show()

# 4. Look at the rows with a blank SeriesDescription.
blank = dicom[dicom['SeriesDescription'].isna()]
print(len(blank), "blank rows")
display(blank[['image_path', 'Rows', 'Columns']].describe())

series UID: 1.3.6.1.4.1.9590.100.1.2.296736403313792599626368780122205399650


,image_path,SeriesDescription,Rows,Columns
8025,CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.296736...,ROI mask images,4808,3024
8026,CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.296736...,cropped images,515,431


ROI mask images 1-250.jpg (3024, 4808) exists: True
cropped images 2-249.jpg (431, 515) exists: True


566 blank rows


,Rows,Columns
count,566.000000,566.000000
mean,5147.722615,3001.005300
std,682.476824,525.110256
min,3880.000000,1801.000000
25%,4584.000000,2682.000000
50%,5063.500000,2958.000000
75%,5491.000000,3168.000000
max,6871.000000,4966.000000


## 4. Open questions / TODO
- [ ] What are the 566 blank-`SeriesDescription` rows? (sizes are large: median about 5063 x 2958, so probably not crops; view a few before deciding)
- [ ] Verify more mappings by eye (not just row 0), including a calc example.
- [ ] Confirm `pathology` values and counts; document `BENIGN_WITHOUT_CALLBACK` -> benign.
- [ ] Check patient overlap between train and test (including across mass/calc).
- [ ] Standardize column names, build one row per ROI crop, save `rois.csv`.